# 9 WorkFlow Analista Senior

### 9.5 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
enriquecer
<br>El Analista Sr corre sus scripts en máquinas virtuales de al menos 128 GB de RAM en Toronto, creando una virtual machine para cada corrida.
<br>Estas virtual machines se auto-suicidarán a los 30 minutos de haber terminado de procesar.

## 9.7  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Sep 03 21:02:26 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671179,35.9,1479894,79.1,1354606,72.4
Vcells,1242556,9.5,8388608,64.0,1978703,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table




Attaching package: ‘data.table’




The following object is masked from ‘package:base’:

    %notin%




Loading required package: R.utils



Loading required package: R.oo



...

The following objects are masked from ‘package:base’:

    attach, detach, load, save




R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.




Attaching package: ‘R.utils’




The following object is masked from ‘package:utils’:

    timestamp




The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, use, warnings




#### Parametros


In [4]:
PARAM <- list()
# semilla tomada de la variable de ambiente SEMILLA
#  si no existe, se usa la semilla del baseline  300089
PARAM$semilla_primigenia <- as.integer(Sys.getenv("SEMILLA", "300089"))

# numero de experimento tomado de la variable de ambiente EXP_NUM
#  es obligatorio: si no existe, se corta la ejecucion
#  (evita pisar por accidente corridas previas con el 9500 base)
if (Sys.getenv("EXP_NUM") == "") {
  stop("Variable de ambiente EXP_NUM no definida (numero de experimento)")
}
PARAM$experimento <- as.integer(Sys.getenv("EXP_NUM"))
PARAM$dataset <- "analistasr_competencia_2026.csv.gz"

# selector de variables FORZADO a canarito (no Boruta, warm-start canarito only)
PARAM$FS$selector <- "canarito"

# parametros de Boruta, tomados de variables de ambiente con defaults
PARAM$BR$ntree         <- as.integer(Sys.getenv("BR_NTREE", "50"))
PARAM$BR$maxRuns       <- as.integer(Sys.getenv("BR_MAXRUNS", "12"))
PARAM$BR$pValue        <- as.numeric(Sys.getenv("BR_PVALUE", "0.05"))
PARAM$BR$mcAdj         <- Sys.getenv("BR_MCADJ", "TRUE") == "TRUE"
PARAM$BR$undersampling <- as.numeric(Sys.getenv("BR_UNDERSAMPLING", "0.10"))
PARAM$BR$holdHistory   <- Sys.getenv("BR_HOLDHISTORY", "TRUE") == "TRUE"
# hilos para ranger; 12 = default para VM de 12 vCPUs (0 = auto/no set)
PARAM$BR$threads       <- as.integer(Sys.getenv("BR_THREADS", "12"))
# pesos de clase para el RF de Boruta; FALSE = version vanilla
#  (Canaritos tampoco usa pesos, solo undersampling)
PARAM$BR$classweights  <- Sys.getenv("BR_CLASSWEIGHTS", "FALSE") == "TRUE"
# Boruta exige maxRuns > 10
if( PARAM$BR$maxRuns <= 10 ) {
  stop("BR_MAXRUNS debe ser mayor a 10 (requisito de Boruta)")
}

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo
#  se agrega la semilla al nombre para que cada corrida independiente
#  tenga su propia carpeta y no se pisen los archivos

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento, "_s", PARAM$semilla_primigenia)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.7.1   Preprocesamiento del dataset

#### 9.7.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.7.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice




Attaching package: ‘mice’




The following object is masked from ‘package:stats’:

    filter




The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.7.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.780952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                       
 [2] "mrentabilidad_annual"                
 [3] "mcomisiones"                         
 [4] "mactivos_margen"                     
 [5] "mpasivos_margen"                     
 [6] "mcuenta_corriente_adicional"         
 [7] "mcuenta_corriente"                   
 [8] "mcaja_ahorro"                        
 [9] "mcaja_ahorro_adicional"              
[10] "mcaja_ahorro_dolares"                
[11] "mcuentas_saldo"                      
[12] "mautoservicio"                       
[13] "mtarjeta_visa_consumo"               
[14] "mtarjeta_master_consumo"             
[15] "mprestamos_personales"               
[16] "mprestamos_prendarios"               
[17] "mprestamos_hipotecarios"             
[18] "mplazo_fijo_dolares"                 
[19] "mplazo_fijo_pesos"                   
[20] "minversion1_pesos"                   
[21] "minversion1_dolares"                 
[22] "minversion2"                         
[23] "mpayroll"                            
[24] "mpayroll2"                           
[25] "mcuenta_debitos_automaticos"         
[26] "mttarjeta_visa_debitos_automaticos"  
[27] "mttarjeta_master_debitos_automaticos"
[28] "mpagodeservicios"                    
[29] "mpagomiscuentas"                     
[30] "mcajeros_propios_descuentos"         
[31] "mtarjeta_visa_descuentos"            
[32] "mtarjeta_master_descuentos"          
[33] "mcomisiones_mantenimiento"           
[34] "mcomisiones_otras"                   
[35] "mforex_buy"                          
[36] "mforex_sell"                         
[37] "mtransferencias_recibidas"           
[38] "mtransferencias_emitidas"            
[39] "mextraccion_autoservicio"            
[40] "mcheques_depositados"                
[41] "mcheques_emitidos"                   
[42] "mcheques_depositados_rechazados"     
[43] "mcheques_emitidos_rechazados"        
[44] "matm"                                
[45] "matm_other"                          
[46] "Master_mfinanciacion_limite"         
[47] "Master_msaldototal"                  
[48] "Master_msaldopesos"                  
[49] "Master_msaldodolares"                
[50] "Master_mconsumospesos"               
[51] "Master_mconsumosdolares"             
[52] "Master_mlimitecompra"                
[53] "Master_madelantopesos"               
[54] "Master_madelantodolares"             
[55] "Master_mpagado"                      
[56] "Master_mpagospesos"                  
[57] "Master_mpagosdolares"                
[58] "Master_mconsumototal"                
[59] "Master_mpagominimo"                  
[60] "Visa_mfinanciacion_limite"           
[61] "Visa_msaldototal"                    
[62] "Visa_msaldopesos"                    
[63] "Visa_msaldodolares"                  
[64] "Visa_mconsumospesos"                 
[65] "Visa_mconsumosdolares"               
[66] "Visa_mlimitecompra"                  
[67] "Visa_madelantopesos"                 
[68] "Visa_madelantodolares"               
[69] "Visa_mpagado"                        
[70] "Visa_mpagospesos"                    
[71] "Visa_mpagosdolares"                  
[72] "Visa_mconsumototal"                  
[73] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "rank_cero_fijo"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_rank_cero_fijo()
mrentabilidad  mrentabilidad_annual  mcomisiones  mactivos_margen  mpasivos_margen  mcuenta_corriente_adicional  mcuenta_corriente  mcaja_ahorro  mcaja_ahorro_adicional  mcaja_ahorro_dolares  mcuentas_saldo  mautoservicio  mtarjeta_visa_consumo  mtarjeta_master_consumo  mprestamos_personales  mprestamos_prendarios  mprestamos_hipotecarios  mplazo_fijo_dolares  mplazo_fijo_pesos  minversion1_pesos  minversion1_dolares  minversion2  mpayroll  mpayroll2  mcuenta_debitos_automaticos  mttarjeta_visa_debitos_automaticos  mttarjeta_master_debitos_automaticos  mpagodeservicios  mpagomiscuentas  mcajeros_propios_descuentos  mtarjeta_visa_descuentos  mtarjeta_master_descuentos  mcomisiones_mantenimiento  mcomisiones_otras  mforex_buy  mforex_sell  mtransferencias_recibidas  mtransferencias_emitidas  mextraccion_autoservicio  mcheques_depositados  mcheques_emitidos  mcheques_depositados_rechazados  mcheques_emitidos_rechazados  matm  matm_other  Master_mfinanciacion_

In [26]:
colnames(dataset)

[1] "numero_de_cliente"                        
  [2] "foto_mes"                                 
  [3] "active_quarter"                           
  [4] "cliente_vip"                              
  [5] "internet"                                 
  [6] "cliente_edad"                             
  [7] "cliente_antiguedad"                       
  [8] "cproductos"                               
  [9] "tcuentas"                                 
 [10] "ccuenta_corriente"                        
 [11] "ccaja_ahorro"                             
 [12] "cdescubierto_preacordado"                 
 [13] "ctarjeta_debito"                          
 [14] "ctarjeta_debito_transacciones"            
 [15] "ctarjeta_visa"                            
 [16] "ctarjeta_visa_transacciones"              
 [17] "ctarjeta_master"                          
 [18] "ctarjeta_master_transacciones"            
 [19] "cprestamos_personales"                    
 [20] "cprestamos_prendarios"                    
 [21] "cprestamos_hipotecarios"                  
 [22] "cplazo_fijo"                              
 [23] "cinversion1"                              
 [24] "cinversion2"                              
 [25] "cseguro_vida"                             
 [26] "cseguro_auto"                             
 [27] "cseguro_vivienda"                         
 [28] "cseguro_accidentes_personales"            
 [29] "ccaja_seguridad"                          
 [30] "cpayroll_trx"                             
 [31] "cpayroll2_trx"                            
 [32] "ccuenta_debitos_automaticos"              
 [33] "ctarjeta_visa_debitos_automaticos"        
 [34] "ctarjeta_master_debitos_automaticos"      
 [35] "cpagodeservicios"                         
 [36] "cpagomiscuentas"                          
 [37] "ccajeros_propios_descuentos"              
 [38] "ctarjeta_visa_descuentos"                 
 [39] "ctarjeta_master_descuentos"               
 [40] "ccomisiones_mantenimiento"                
 [41] "ccomisiones_otras"                        
 [42] "cforex"                                   
 [43] "cforex_buy"                               
 [44] "cforex_sell"                              
 [45] "ctransferencias_recibidas"                
 [46] "ctransferencias_emitidas"                 
 [47] "cextraccion_autoservicio"                 
 [48] "ccheques_depositados"                     
 [49] "ccheques_emitidos"                        
 [50] "ccheques_depositados_rechazados"          
 [51] "ccheques_emitidos_rechazados"             
 [52] "tcallcenter"                              
 [53] "ccallcenter_transacciones"                
 [54] "thomebanking"                             
 [55] "chomebanking_transacciones"               
 [56] "ccajas_transacciones"                     
 [57] "ccajas_consultas"                         
 [58] "ccajas_depositos"                         
 [59] "ccajas_extracciones"                      
 [60] "ccajas_otras"                             
 [61] "catm_trx"                                 
 [62] "catm_trx_other"                           
 [63] "ctrx_quarter"                             
 [64] "tmobile_app"                              
 [65] "cmobile_app_trx"                          
 [66] "Master_delinquency"                       
 [67] "Master_status"                            
 [68] "Master_Fvencimiento"                      
 [69] "Master_Finiciomora"                       
 [70] "Master_fultimo_cierre"                    
 [71] "Master_fechaalta"                         
 [72] "Master_cconsumos"                         
 [73] "Master_cadelantosefectivo"                
 [74] "Visa_delinquency"                         
 [75] "Visa_status"                              
 [76] "Visa_Fvencimiento"                        
 [77] "Visa_Finiciomora"                         
 [78] "Visa_fultimo_cierre"                      
 [79] "Visa_fechaalta"                           
 [80] "Visa_cconsumos"                           
 [

#### 9.7.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [27]:
if( !require("lubridate")) install.packages("lubridate", repos = "http://cran.us.r-project.org")
require("lubridate")

Loading required package: lubridate




Attaching package: ‘lubridate’




The following objects are masked from ‘package:data.table’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year




The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [28]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

In [29]:
# Esta es la parte que los alumnos deben desplegar todo su ingenio
# Agregar aqui sus PROPIAS VARIABLES manuales

AgregarVariables_IntraMes <- function(dataset) {
  cat( "inicio AgregarVariables_IntraMes()\n")
  gc(verbose= FALSE)
  # INICIO de la seccion donde se deben hacer cambios con variables nuevas

  # el mes 1,2, ..12
  if( atributos_presentes( c("foto_mes") ))
    dataset[, kmes := foto_mes %% 100]

  # creo un ctr_quarter que tenga en cuenta cuando
  # los clientes hace 3 menos meses que estan
  # ya que seria injusto considerar las transacciones medidas en menor tiempo
  if( atributos_presentes( c("ctrx_quarter") ))
    dataset[, ctrx_quarter_normalizado := as.numeric(ctrx_quarter) ]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 1, ctrx_quarter_normalizado := ctrx_quarter * 5]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 2, ctrx_quarter_normalizado := ctrx_quarter * 2]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[
      cliente_antiguedad == 3,
      ctrx_quarter_normalizado := ctrx_quarter * 1.2
    ]

   if(atributos_presentes(c("foto_mes")))
    dataset[,foto_mes_formato_fecha := as.Date(paste(substr(dataset$foto_mes,1,4),substr(dataset$foto_mes,5,6),"01",sep='-'))]

  #dataset$foto_mes_formato_fecha <<- as.Date(paste(substr(dataset$foto_mes,1,4),substr(dataset$foto_mes,5,6),"01",sep='-'))

  if(atributos_presentes(c("cantidad_total_transacciones"))){
   auxiliarmenos1 <- dataset[,list(numero_de_cliente,foto_mes_formato_fecha, cantidad_total_transacciones)]
   auxiliarmenos2 <- dataset[,list(numero_de_cliente,foto_mes_formato_fecha,cantidad_total_transacciones)]
   # auxiliarmenos1$foto_mes_formato_fecha <- as.Date(auxiliarmenos1$foto_mes_formato_fecha)
   # auxiliarmenos2$foto_mes_formato_fecha <- as.Date(auxiliarmenos2$foto_mes_formato_fecha)
   auxiliarmenos1$foto_mes_formato_fecha <- auxiliarmenos1$foto_mes_formato_fecha  %m-%  months(1)
   auxiliarmenos2$foto_mes_formato_fecha <- auxiliarmenos2$foto_mes_formato_fecha %m-% months(2)
   auxiliarmenos1$codigo <- paste(auxiliarmenos1$numero_de_cliente,auxiliarmenos1$foto_mes_formato_fecha,sep='-')
   auxiliarmenos2$codigo <- paste(auxiliarmenos2$numero_de_cliente,auxiliarmenos2$foto_mes_formato_fecha,sep='-')

   dataset[, codigo := paste(numero_de_cliente, foto_mes_formato_fecha, sep='-') ]

   dataset[ auxiliarmenos1,
            on = "codigo",
            transaccionesmenos1 := i.cantidad_total_transacciones ]

   dataset[ auxiliarmenos2,
            on = "codigo",
            transaccionesmenos2 := i.cantidad_total_transacciones ]

   dataset[, cantidad_total_transacciones_quarter := rowSums(cbind(cantidad_total_transacciones +
    transaccionesmenos1 + transaccionesmenos2),na.rm=T) ]

   dataset[, codigo := NULL ]
   dataset[, transaccionesmenos1 := NULL ]
   dataset[, transaccionesmenos2 := NULL ]
   dataset[, foto_mes_formato_fecha := NULL ]
   rm(auxiliarmenos1)
   rm(auxiliarmenos2)
  }

  if( atributos_presentes( c("cantidad_total_transacciones_quarter") ))
    dataset[, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter]

  if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 1, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 5]

  if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 2, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 2]

  if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 3, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 1.2]

  # variable extraida de una tesis de maestria de Irlanda
  if( atributos_presentes( c("mpayroll", "cliente_edad") ))
    dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

  # se crean los nuevos campos para MasterCard  y Visa,
  #  teniendo en cuenta los NA's
  # varias formas de combinar Visa_status y Master_status
  if( atributos_presentes( c("Master_status", "Visa_status") ))
  {
    dataset[, vm_status01 := pmax(Master_status, Visa_status, na.rm = TRUE)]
    dataset[, vm_status02 := Master_status + Visa_status]

    dataset[, vm_status03 := pmax(
      ifelse(is.na(Master_status), 10, Master_status),
      ifelse(is.na(Visa_status), 10, Visa_status)
    )]

    dataset[, vm_status04 := ifelse(is.na(Master_status), 10, Master_status)
      + ifelse(is.na(Visa_status), 10, Visa_status)]

    dataset[, vm_status05 := ifelse(is.na(Master_status), 10, Master_status)
      + 100 * ifelse(is.na(Visa_status), 10, Visa_status)]

    dataset[, vm_status06 := ifelse(is.na(Visa_status),
      ifelse(is.na(Master_status), 10, Master_status),
      Visa_status
    )]

    dataset[, mv_status07 := ifelse(is.na(Master_status),
      ifelse(is.na(Visa_status), 10, Visa_status),
      Master_status
    )]
  }


  # combino MasterCard y Visa
  if( atributos_presentes( c("Master_mfinanciacion_limite", "Visa_mfinanciacion_limite") ))
    dataset[, vm_mfinanciacion_limite := rowSums(cbind(Master_mfinanciacion_limite, Visa_mfinanciacion_limite), na.rm = TRUE)]

  if( atributos_presentes( c("Master_Fvencimiento", "Visa_Fvencimiento") ))
    dataset[, vm_Fvencimiento := pmin(Master_Fvencimiento, Visa_Fvencimiento, na.rm = TRUE)]

  if( atributos_presentes( c("Master_Finiciomora", "Visa_Finiciomora") ))
    dataset[, vm_Finiciomora := pmin(Master_Finiciomora, Visa_Finiciomora, na.rm = TRUE)]

  if( atributos_presentes( c("Master_msaldototal", "Visa_msaldototal") ))
    dataset[, vm_msaldototal := rowSums(cbind(Master_msaldototal, Visa_msaldototal), na.rm = TRUE)]

  if( atributos_presentes( c("Master_msaldopesos", "Visa_msaldopesos") ))
    dataset[, vm_msaldopesos := rowSums(cbind(Master_msaldopesos, Visa_msaldopesos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_msaldodolares", "Visa_msaldodolares") ))
    dataset[, vm_msaldodolares := rowSums(cbind(Master_msaldodolares, Visa_msaldodolares), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mconsumospesos", "Visa_mconsumospesos") ))
    dataset[, vm_mconsumospesos := rowSums(cbind(Master_mconsumospesos, Visa_mconsumospesos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mconsumosdolares", "Visa_mconsumosdolares") ))
    dataset[, vm_mconsumosdolares := rowSums(cbind(Master_mconsumosdolares, Visa_mconsumosdolares), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mlimitecompra", "Visa_mlimitecompra") ))
    dataset[, vm_mlimitecompra := rowSums(cbind(Master_mlimitecompra, Visa_mlimitecompra), na.rm = TRUE)]

  if( atributos_presentes( c("Master_madelantopesos", "Visa_madelantopesos") ))
    dataset[, vm_madelantopesos := rowSums(cbind(Master_madelantopesos, Visa_madelantopesos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_madelantodolares", "Visa_madelantodolares") ))
    dataset[, vm_madelantodolares := rowSums(cbind(Master_madelantodolares, Visa_madelantodolares), na.rm = TRUE)]

  if( atributos_presentes( c("Master_fultimo_cierre", "Visa_fultimo_cierre") ))
    dataset[, vm_fultimo_cierre := pmax(Master_fultimo_cierre, Visa_fultimo_cierre, na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagado", "Visa_mpagado") ))
    dataset[, vm_mpagado := rowSums(cbind(Master_mpagado, Visa_mpagado), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagospesos", "Visa_mpagospesos") ))
    dataset[, vm_mpagospesos := rowSums(cbind(Master_mpagospesos, Visa_mpagospesos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagosdolares", "Visa_mpagosdolares") ))
    dataset[, vm_mpagosdolares := rowSums(cbind(Master_mpagosdolares, Visa_mpagosdolares), na.rm = TRUE)]

  if( atributos_presentes( c("Master_fechaalta", "Visa_fechaalta") ))
    dataset[, vm_fechaalta := pmax(Master_fechaalta, Visa_fechaalta, na.rm = TRUE)]

  if( atributos_presentes( c("Master_mconsumototal", "Visa_mconsumototal") ))
    dataset[, vm_mconsumototal := rowSums(cbind(Master_mconsumototal, Visa_mconsumototal), na.rm = TRUE)]

  if( atributos_presentes( c("Master_cconsumos", "Visa_cconsumos") ))
    dataset[, vm_cconsumos := rowSums(cbind(Master_cconsumos, Visa_cconsumos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_cadelantosefectivo", "Visa_cadelantosefectivo") ))
    dataset[, vm_cadelantosefectivo := rowSums(cbind(Master_cadelantosefectivo, Visa_cadelantosefectivo), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagominimo", "Visa_mpagominimo") ))
    dataset[, vm_mpagominimo := rowSums(cbind(Master_mpagominimo, Visa_mpagominimo), na.rm = TRUE)]

  # a partir de aqui juego con la suma de Mastercard y Visa
  if( atributos_presentes( c("Master_mlimitecompra", "vm_mlimitecompra") ))
    dataset[, vmr_Master_mlimitecompra := Master_mlimitecompra / vm_mlimitecompra]

  if( atributos_presentes( c("Visa_mlimitecompra", "vm_mlimitecompra") ))
    dataset[, vmr_Visa_mlimitecompra := Visa_mlimitecompra / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldototal", "vm_mlimitecompra") ))
    dataset[, vmr_msaldototal := vm_msaldototal / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldopesos", "vm_mlimitecompra") ))
    dataset[, vmr_msaldopesos := vm_msaldopesos / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldopesos", "vm_msaldototal") ))
    dataset[, vmr_msaldopesos2 := vm_msaldopesos / vm_msaldototal]

  if( atributos_presentes( c("vm_msaldodolares", "vm_mlimitecompra") ))
    dataset[, vmr_msaldodolares := vm_msaldodolares / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldodolares", "vm_msaldototal") ))
    dataset[, vmr_msaldodolares2 := vm_msaldodolares / vm_msaldototal]

  if( atributos_presentes( c("vm_mconsumospesos", "vm_mlimitecompra") ))
    dataset[, vmr_mconsumospesos := vm_mconsumospesos / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mconsumosdolares", "vm_mlimitecompra") ))
    dataset[, vmr_mconsumosdolares := vm_mconsumosdolares / vm_mlimitecompra]

  if( atributos_presentes( c("vm_madelantopesos", "vm_mlimitecompra") ))
    dataset[, vmr_madelantopesos := vm_madelantopesos / vm_mlimitecompra]

  if( atributos_presentes( c("vm_madelantodolares", "vm_mlimitecompra") ))
    dataset[, vmr_madelantodolares := vm_madelantodolares / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagado", "vm_mlimitecompra") ))
    dataset[, vmr_mpagado := vm_mpagado / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagospesos", "vm_mlimitecompra") ))
    dataset[, vmr_mpagospesos := vm_mpagospesos / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagosdolares", "vm_mlimitecompra") ))
    dataset[, vmr_mpagosdolares := vm_mpagosdolares / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mconsumototal", "vm_mlimitecompra") ))
    dataset[, vmr_mconsumototal := vm_mconsumototal / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagominimo", "vm_mlimitecompra") ))
    dataset[, vmr_mpagominimo := vm_mpagominimo / vm_mlimitecompra]

  # Aqui debe usted agregar sus propias nuevas variables

  # valvula de seguridad para evitar valores infinitos
  # paso los infinitos a NULOS
  infinitos <- lapply(
    names(dataset),
    function(.name) dataset[, sum(is.infinite(get(.name)))]
  )

  infinitos_qty <- sum(unlist(infinitos))
  if (infinitos_qty > 0) {
    cat(
      "ATENCION, hay", infinitos_qty,
      "valores infinitos en tu dataset. Seran pasados a NA\n"
    )
    dataset[mapply(is.infinite, dataset)] <<- NA
  }


  # valvula de seguridad para evitar valores NaN  que es 0/0
  # paso los NaN a 0 , decision polemica si las hay
  # se invita a asignar un valor razonable segun la semantica del campo creado
  nans <- lapply(
    names(dataset),
    function(.name) dataset[, sum(is.nan(get(.name)))]
  )

  nans_qty <- sum(unlist(nans))
  if (nans_qty > 0) {
    cat(
      "ATENCION, hay", nans_qty,
      "valores NaN 0/0 en tu dataset. Seran pasados arbitrariamente a 0\n"
    )

    cat("Si no te gusta la decision, modifica a gusto el programa!\n\n")
    dataset[mapply(is.nan, dataset)] <<- 0
  }

  cat( "fin AgregarVariables_IntraMes()\n")
}


In [30]:
# agrego las variables intra mes

AgregarVariables_IntraMes(dataset)

inicio AgregarVariables_IntraMes()
fin AgregarVariables_IntraMes()


In [31]:
# visualizo las columas del dataset a esta etapa
ncol(dataset)
colnames(dataset)

[1] 171

[1] "numero_de_cliente"                        
  [2] "foto_mes"                                 
  [3] "active_quarter"                           
  [4] "cliente_vip"                              
  [5] "internet"                                 
  [6] "cliente_edad"                             
  [7] "cliente_antiguedad"                       
  [8] "cproductos"                               
  [9] "tcuentas"                                 
 [10] "ccuenta_corriente"                        
 [11] "ccaja_ahorro"                             
 [12] "cdescubierto_preacordado"                 
 [13] "ctarjeta_debito"                          
 [14] "ctarjeta_debito_transacciones"            
 [15] "ctarjeta_visa"                            
 [16] "ctarjeta_visa_transacciones"              
 [17] "ctarjeta_master"                          
 [18] "ctarjeta_master_transacciones"            
 [19] "cprestamos_personales"                    
 [20] "cprestamos_prendarios"                    
 [21] "cprestamos_hipotecarios"                  
 [22] "cplazo_fijo"                              
 [23] "cinversion1"                              
 [24] "cinversion2"                              
 [25] "cseguro_vida"                             
 [26] "cseguro_auto"                             
 [27] "cseguro_vivienda"                         
 [28] "cseguro_accidentes_personales"            
 [29] "ccaja_seguridad"                          
 [30] "cpayroll_trx"                             
 [31] "cpayroll2_trx"                            
 [32] "ccuenta_debitos_automaticos"              
 [33] "ctarjeta_visa_debitos_automaticos"        
 [34] "ctarjeta_master_debitos_automaticos"      
 [35] "cpagodeservicios"                         
 [36] "cpagomiscuentas"                          
 [37] "ccajeros_propios_descuentos"              
 [38] "ctarjeta_visa_descuentos"                 
 [39] "ctarjeta_master_descuentos"               
 [40] "ccomisiones_mantenimiento"                
 [41] "ccomisiones_otras"                        
 [42] "cforex"                                   
 [43] "cforex_buy"                               
 [44] "cforex_sell"                              
 [45] "ctransferencias_recibidas"                
 [46] "ctransferencias_emitidas"                 
 [47] "cextraccion_autoservicio"                 
 [48] "ccheques_depositados"                     
 [49] "ccheques_emitidos"                        
 [50] "ccheques_depositados_rechazados"          
 [51] "ccheques_emitidos_rechazados"             
 [52] "tcallcenter"                              
 [53] "ccallcenter_transacciones"                
 [54] "thomebanking"                             
 [55] "chomebanking_transacciones"               
 [56] "ccajas_transacciones"                     
 [57] "ccajas_consultas"                         
 [58] "ccajas_depositos"                         
 [59] "ccajas_extracciones"                      
 [60] "ccajas_otras"                             
 [61] "catm_trx"                                 
 [62] "catm_trx_other"                           
 [63] "ctrx_quarter"                             
 [64] "tmobile_app"                              
 [65] "cmobile_app_trx"                          
 [66] "Master_delinquency"                       
 [67] "Master_status"                            
 [68] "Master_Fvencimiento"                      
 [69] "Master_Finiciomora"                       
 [70] "Master_fultimo_cierre"                    
 [71] "Master_fechaalta"                         
 [72] "Master_cconsumos"                         
 [73] "Master_cadelantosefectivo"                
 [74] "Visa_delinquency"                         
 [75] "Visa_status"                              
 [76] "Visa_Fvencimiento"                        
 [77] "Visa_Finiciomora"                         
 [78] "Visa_fultimo_cierre"                      
 [79] "Visa_fechaalta"                           
 [80] "Visa_cconsumos"                           
 [

#### 9.7.1.4  FEhist Feature Engineering historico

El Feature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [32]:
if( !require("Rcpp")) install.packages("Rcpp", repos = "http://cran.us.r-project.org")
require("Rcpp")

Loading required package: Rcpp



In [33]:
# se calculan para los 6 meses previos el minimo, maximo y
#  tendencia calculada con cuadrados minimos
# la formula de calculo de la tendencia puede verse en
#  https://stats.libretexts.org/Bookshelves/Introductory_Statistics/Book%3A_Introductory_Statistics_(Shafer_and_Zhang)/10%3A_Correlation_and_Regression/10.04%3A_The_Least_Squares_Regression_Line
# para la maxíma velocidad esta funcion esta escrita en lenguaje C,
# y no en la porqueria de R o Python

cppFunction("NumericVector fhistC(NumericVector pcolumna, IntegerVector pdesde )
{
  /* Aqui se cargan los valores para la regresion */
  double  x[100] ;
  double  y[100] ;

  int n = pcolumna.size();
  NumericVector out( 5*n );

  for(int i = 0; i < n; i++)
  {
    //lag
    if( pdesde[i]-1 < i )  out[ i + 4*n ]  =  pcolumna[i-1] ;
    else                   out[ i + 4*n ]  =  NA_REAL ;


    int  libre    = 0 ;
    int  xvalor   = 1 ;

    for( int j= pdesde[i]-1;  j<=i; j++ )
    {
       double a = pcolumna[j] ;

       if( !R_IsNA( a ) )
       {
          y[ libre ]= a ;
          x[ libre ]= xvalor ;
          libre++ ;
       }

       xvalor++ ;
    }

    /* Si hay al menos dos valores */
    if( libre > 1 )
    {
      double  xsum  = x[0] ;
      double  ysum  = y[0] ;
      double  xysum = xsum * ysum ;
      double  xxsum = xsum * xsum ;
      double  vmin  = y[0] ;
      double  vmax  = y[0] ;

      for( int h=1; h<libre; h++)
      {
        xsum  += x[h] ;
        ysum  += y[h] ;
        xysum += x[h]*y[h] ;
        xxsum += x[h]*x[h] ;

        if( y[h] < vmin )  vmin = y[h] ;
        if( y[h] > vmax )  vmax = y[h] ;
      }

      out[ i ]  =  (libre*xysum - xsum*ysum)/(libre*xxsum -xsum*xsum) ;
      out[ i + n ]    =  vmin ;
      out[ i + 2*n ]  =  vmax ;
      out[ i + 3*n ]  =  ysum / libre ;
    }
    else
    {
      out[ i       ]  =  NA_REAL ;
      out[ i + n   ]  =  NA_REAL ;
      out[ i + 2*n ]  =  NA_REAL ;
      out[ i + 3*n ]  =  NA_REAL ;
    }
  }

  return  out;
}")


In [34]:
# calcula la tendencia de las variables cols de los ultimos 6 meses
# la tendencia es la pendiente de la recta que ajusta por cuadrados minimos
# La funcionalidad de ratioavg es autoria de  Daiana Sparta,  UAustral  2021

TendenciaYmuchomas <- function(
    dataset, cols, ventana = 6, tendencia = TRUE,
    minimo = TRUE, maximo = TRUE, promedio = TRUE,
    ratioavg = FALSE, ratiomax = FALSE) {
  gc(verbose= FALSE)
  # Esta es la cantidad de meses que utilizo para la historia
  ventana_regresion <- ventana

  last <- nrow(dataset)

  # creo el vector_desde que indica cada ventana
  # de esta forma se acelera el procesamiento ya que lo hago una sola vez
  vector_ids <- dataset[ , numero_de_cliente ]

  vector_desde <- seq(
    -ventana_regresion + 2,
    nrow(dataset) - ventana_regresion + 1
  )

  vector_desde[1:ventana_regresion] <- 1

  for (i in 2:last) {
    if (vector_ids[i - 1] != vector_ids[i]) {
      vector_desde[i] <- i
    }
  }
  for (i in 2:last) {
    if (vector_desde[i] < vector_desde[i - 1]) {
      vector_desde[i] <- vector_desde[i - 1]
    }
  }

  for (campo in cols) {
    nueva_col <- fhistC(dataset[, get(campo)], vector_desde)

    if (tendencia) {
      dataset[, paste0(campo, "_tend", ventana) :=
        nueva_col[(0 * last + 1):(1 * last)]]
    }

    if (minimo) {
      dataset[, paste0(campo, "_min", ventana) :=
        nueva_col[(1 * last + 1):(2 * last)]]
    }

    if (maximo) {
      dataset[, paste0(campo, "_max", ventana) :=
        nueva_col[(2 * last + 1):(3 * last)]]
    }

    if (promedio) {
      dataset[, paste0(campo, "_avg", ventana) :=
        nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratioavg) {
      dataset[, paste0(campo, "_ratioavg", ventana) :=
        get(campo) / nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratiomax) {
      dataset[, paste0(campo, "_ratiomax", ventana) :=
        get(campo) / nueva_col[(2 * last + 1):(3 * last)]]
    }
  }
}


In [35]:
# Feature Engineering Historico

setorder(dataset, numero_de_cliente, foto_mes)

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


In [36]:
# parametros de Feature Engineering Historico de Tendencias
PARAM$FE_hist$Tendencias$run <- TRUE
PARAM$FE_hist$Tendencias$ventana <- 6
PARAM$FE_hist$Tendencias$tendencia <- TRUE
PARAM$FE_hist$Tendencias$minimo <- FALSE
PARAM$FE_hist$Tendencias$maximo <- FALSE
PARAM$FE_hist$Tendencias$promedio <- FALSE
PARAM$FE_hist$Tendencias$ratioavg <- FALSE
PARAM$FE_hist$Tendencias$ratiomax <- FALSE


In [37]:
cols_lagueables <- intersect(cols_lagueables, colnames(dataset))
setorder(dataset, numero_de_cliente, foto_mes)

if( PARAM$FE_hist$Tendencias$run) {
    TendenciaYmuchomas(dataset,
    cols = cols_lagueables,
    ventana = PARAM$FE_hist$Tendencias$ventana, # 6 meses de historia
    tendencia = PARAM$FE_hist$Tendencias$tendencia,
    minimo = PARAM$FE_hist$Tendencias$minimo,
    maximo = PARAM$FE_hist$Tendencias$maximo,
    promedio = PARAM$FE_hist$Tendencias$promedio,
    ratioavg = PARAM$FE_hist$Tendencias$ratioavg,
    ratiomax = PARAM$FE_hist$Tendencias$ratiomax
  )
}



Verificacion de los campos recien creados

In [38]:
ncol(dataset)
colnames(dataset)

[1] 1011

[1] "numero_de_cliente"                               
   [2] "foto_mes"                                        
   [3] "active_quarter"                                  
   [4] "cliente_vip"                                     
   [5] "internet"                                        
   [6] "cliente_edad"                                    
   [7] "cliente_antiguedad"                              
   [8] "cproductos"                                      
   [9] "tcuentas"                                        
  [10] "ccuenta_corriente"                               
  [11] "ccaja_ahorro"                                    
  [12] "cdescubierto_preacordado"                        
  [13] "ctarjeta_debito"                                 
  [14] "ctarjeta_debito_transacciones"                   
  [15] "ctarjeta_visa"                                   
  [16] "ctarjeta_visa_transacciones"                     
  [17] "ctarjeta_master"                                 
  [18] "ctarjeta_master_transacciones"                   
  [19] "cprestamos_personales"                           
  [20] "cprestamos_prendarios"                           
  [21] "cprestamos_hipotecarios"                         
  [22] "cplazo_fijo"                                     
  [23] "cinversion1"                                     
  [24] "cinversion2"                                     
  [25] "cseguro_vida"                                    
  [26] "cseguro_auto"                                    
  [27] "cseguro_vivienda"                                
  [28] "cseguro_accidentes_personales"                   
  [29] "ccaja_seguridad"                                 
  [30] "cpayroll_trx"                                    
  [31] "cpayroll2_trx"                                   
  [32] "ccuenta_debitos_automaticos"                     
  [33] "ctarjeta_visa_debitos_automaticos"               
  [34] "ctarjeta_master_debitos_automaticos"             
  [35] "cpagodeservicios"                                
  [36] "cpagomiscuentas"                                 
  [37] "ccajeros_propios_descuentos"                     
  [38] "ctarjeta_visa_descuentos"                        
  [39] "ctarjeta_master_descuentos"                      
  [40] "ccomisiones_mantenimiento"                       
  [41] "ccomisiones_otras"                               
  [42] "cforex"                                          
  [43] "cforex_buy"                                      
  [44] "cforex_sell"                                     
  [45] "ctransferencias_recibidas"                       
  [46] "ctransferencias_emitidas"                        
  [47] "cextraccion_autoservicio"                        
  [48] "ccheques_depositados"                            
  [49] "ccheques_emitidos"                               
  [50] "ccheques_depositados_rechazados"                 
  [51] "ccheques_emitidos_rechazados"                    
  [52] "tcallcenter"                                     
  [53] "ccallcenter_transacciones"                       
  [54] "thomebanking"                                    
  [55] "chomebanking_transacciones"                      
  [56] "ccajas_transacciones"                            
  [57] "ccajas_consultas"                                
  [58] "ccajas_depositos"                                
  [59] "ccajas_extracciones"                             
  [60] "ccajas_otras"                                    
  [61] "catm_trx"                                        
  [62] "catm_trx_other"                                  
  [63] "ctrx_quarter"                                    
  [64] "tmobile_app"                                     
  [65] "cmobile_app_trx"                                 
  [66] "Master_delinquency"                              
  [67] "Master_status"                                   
  [68] "Master_Fvencimiento"                             
  [69] "Master_Finiciomora"                              
 

#### 9.7.1.5  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest



In [39]:
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

Loading required package: lightgbm



In [40]:
AgregaVarRandomForest <- function() {

  cat( "inicio AgregaVarRandomForest()\n")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[ clase_ternaria %in% PARAM$FE_rf$train$clase01_valor1,
      clase01 := 1L ]

  campos_buenos <- setdiff(
    colnames(dataset),
    c( "clase_ternaria", "clase01")
  )

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$FE_rf$train$training )]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    free_raw_data = FALSE
  )

  modelo <- lgb.train(
     data = dtrain,
     param = PARAM$FE_rf$lgb_param,
     verbose = -100
  )

  cat( "Fin construccion RandomForest\n" )
  # grabo el modelo, achivo .model
  lgb.save(modelo, file="modelo.model" )

  qarbolitos <- copy(PARAM$FE_rf$lgb_param$num_iterations)

  periodos <- dataset[ , unique( foto_mes ) ]

  for( periodo in  periodos )
  {
    cat( "periodo = ", periodo, "\n" )
    datamatrix <- data.matrix(dataset[ foto_mes== periodo, campos_buenos, with = FALSE])

    cat( "Inicio prediccion\n" )
    prediccion <- predict(
        modelo,
        datamatrix,
        type = "leaf"
    )
    cat( "Fin prediccion\n" )

    for( arbolito in 1:qarbolitos )
    {
       cat( arbolito, " " )
       hojas_arbol <- unique(prediccion[ , arbolito])

       for (pos in 1:length(hojas_arbol)) {
         # el numero de nodo de la hoja, estan salteados
         nodo_id <- hojas_arbol[pos]
         dataset[ foto_mes== periodo, paste0(
            "rf_", sprintf("%03d", arbolito),
             "_", sprintf("%03d", nodo_id)
          ) :=  as.integer( nodo_id == prediccion[ , arbolito]) ]

       }

       rm( hojas_arbol )
    }
    cat( "\n" )

    rm( prediccion )
    rm( datamatrix )
    gc(verbose= FALSE)
  }

  gc(verbose= FALSE)

  # borro clase01 , no debe ensuciar el dataset
  dataset[ , clase01 := NULL ]

}


In [41]:
# Parametros de Feature Engineering  a partir de hojas de Random Forest

# Estos CUATRO parametros son los que se deben modificar
PARAM$FE_rf$arbolitos= 20
PARAM$FE_rf$hojas_por_arbol= 16
PARAM$FE_rf$datos_por_hoja= 100
PARAM$FE_rf$mtry_ratio= 0.2

# Estos son quasi fijos
PARAM$FE_rf$train$clase01_valor1 <- c( "BAJA+2", "BAJA+1")
PARAM$FE_rf$train$training <- c( 202101, 202102, 202103)

# Estos TAMBIEN son quasi fijos
PARAM$FE_rf$lgb_param <-list(
    # parametros que se pueden cambiar
    num_iterations = PARAM$FE_rf$arbolitos,
    num_leaves  = PARAM$FE_rf$hojas_por_arbol,
    min_data_in_leaf = PARAM$FE_rf$datos_por_hoja,
    feature_fraction_bynode  = PARAM$FE_rf$mtry_ratio,

    # para que LightGBM emule Random Forest
    boosting = "rf",
    bagging_fraction = ( 1.0 - 1.0/exp(1.0) ),
    bagging_freq = 1.0,
    feature_fraction = 1.0,

    # genericos de LightGBM
    max_bin = 31L,
    objective = "binary",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    force_row_wise = TRUE,
    verbosity = -100,
    max_depth = -1L,
    min_gain_to_split = 0.0,
    min_sum_hessian_in_leaf = 0.001,
    lambda_l1 = 0.0,
    lambda_l2 = 0.0,

    pos_bagging_fraction = 1.0,
    neg_bagging_fraction = 1.0,
    is_unbalance = FALSE,
    scale_pos_weight = 1.0,

    drop_rate = 0.1,
    max_drop = 50,
    skip_drop = 0.5,

    extra_trees = FALSE
  )

In [42]:
# Feature Engineering agregando variables de Random Forest
AgregaVarRandomForest()

inicio AgregaVarRandomForest()
Fin construccion RandomForest
periodo =  201901 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201902 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201903 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201904 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201905 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201906 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201907 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  201908 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7

In [43]:
ncol(dataset)
colnames(dataset)

[1] 1332

[1] "numero_de_cliente"                               
   [2] "foto_mes"                                        
   [3] "active_quarter"                                  
   [4] "cliente_vip"                                     
   [5] "internet"                                        
   [6] "cliente_edad"                                    
   [7] "cliente_antiguedad"                              
   [8] "cproductos"                                      
   [9] "tcuentas"                                        
  [10] "ccuenta_corriente"                               
  [11] "ccaja_ahorro"                                    
  [12] "cdescubierto_preacordado"                        
  [13] "ctarjeta_debito"                                 
  [14] "ctarjeta_debito_transacciones"                   
  [15] "ctarjeta_visa"                                   
  [16] "ctarjeta_visa_transacciones"                     
  [17] "ctarjeta_master"                                 
  [18] "ctarjeta_master_transacciones"                   
  [19] "cprestamos_personales"                           
  [20] "cprestamos_prendarios"                           
  [21] "cprestamos_hipotecarios"                         
  [22] "cplazo_fijo"                                     
  [23] "cinversion1"                                     
  [24] "cinversion2"                                     
  [25] "cseguro_vida"                                    
  [26] "cseguro_auto"                                    
  [27] "cseguro_vivienda"                                
  [28] "cseguro_accidentes_personales"                   
  [29] "ccaja_seguridad"                                 
  [30] "cpayroll_trx"                                    
  [31] "cpayroll2_trx"                                   
  [32] "ccuenta_debitos_automaticos"                     
  [33] "ctarjeta_visa_debitos_automaticos"               
  [34] "ctarjeta_master_debitos_automaticos"             
  [35] "cpagodeservicios"                                
  [36] "cpagomiscuentas"                                 
  [37] "ccajeros_propios_descuentos"                     
  [38] "ctarjeta_visa_descuentos"                        
  [39] "ctarjeta_master_descuentos"                      
  [40] "ccomisiones_mantenimiento"                       
  [41] "ccomisiones_otras"                               
  [42] "cforex"                                          
  [43] "cforex_buy"                                      
  [44] "cforex_sell"                                     
  [45] "ctransferencias_recibidas"                       
  [46] "ctransferencias_emitidas"                        
  [47] "cextraccion_autoservicio"                        
  [48] "ccheques_depositados"                            
  [49] "ccheques_emitidos"                               
  [50] "ccheques_depositados_rechazados"                 
  [51] "ccheques_emitidos_rechazados"                    
  [52] "tcallcenter"                                     
  [53] "ccallcenter_transacciones"                       
  [54] "thomebanking"                                    
  [55] "chomebanking_transacciones"                      
  [56] "ccajas_transacciones"                            
  [57] "ccajas_consultas"                                
  [58] "ccajas_depositos"                                
  [59] "ccajas_extracciones"                             
  [60] "ccajas_otras"                                    
  [61] "catm_trx"                                        
  [62] "catm_trx_other"                                  
  [63] "ctrx_quarter"                                    
  [64] "tmobile_app"                                     
  [65] "cmobile_app_trx"                                 
  [66] "Master_delinquency"                              
  [67] "Master_status"                                   
  [68] "Master_Fvencimiento"                             
  [69] "Master_Finiciomora"                              
 

#### 9.7.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr*

El objetivo de esta etapa NO es mejorar el modelo predictivo

El objetivo es eliminar campos poco importantes para hacer espacio a nuevos campos, debido a las restricciones de memoria RAM.

In [44]:
VPOS_CORTE <- c()

fganancia_lgbm_meseta <- function(probs, datos) {
  vlabels <- get_field(datos, "label")
  vpesos <- get_field(datos, "weight")

  tbl <- as.data.table(list(
    "prob" = probs,
    "gan" = ifelse(vlabels == 1 & vpesos > 1, PARAM$CN$train$gan1, PARAM$CN$train$gan0)
  ))

  setorder(tbl, -prob)
  tbl[, posicion := .I]
  tbl[, gan_acum := cumsum(gan)]
  setorder(tbl, -gan_acum) # voy por la meseta

  gan <- mean(tbl[1:500, gan_acum]) # meseta de tamaño 500

  pos_meseta <- tbl[1:500, median(posicion)]
  VPOS_CORTE <<- c(VPOS_CORTE, pos_meseta)

  return(list(
    "name" = "ganancia",
    "value" = gan,
    "higher_better" = TRUE
  ))
}


In [45]:
# Elimina del dataset las variables que estan por debajo
#  de la capa geologica de canaritos
# se llama varias veces, luego de agregar muchas variables nuevas,
#  para ir reduciendo la cantidad de variables
# y así hacer lugar a nuevas variables importantes

GVEZ <- 1

campitos <- c( "numero_de_cliente", "foto_mes", "clase_ternaria" )

CanaritosAsesinos <- function(
  canaritos_ratio,
  canaritos_desvios,
  canaritos_semilla) {

  cat( "inicio CanaritosAsesinos()\n")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[ clase_ternaria %in% PARAM$CN$train$clase01_valor1,
      clase01 := 1L ]

  set.seed(canaritos_semilla, kind = "L'Ecuyer-CMRG")
  for (i in 1:(ncol(dataset) * canaritos_ratio)) {
    dataset[, paste0("canarito", i) := runif(nrow(dataset))]
  }

  campos_buenos <- setdiff(
    colnames(dataset),
    c( campitos, "clase01")
  )

  azar <- runif(nrow(dataset))

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$CN$train$training &
      (clase01 == 1 | azar < PARAM$CN$train$undersampling))]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    weight = dataset[
      entrenamiento == TRUE,
      ifelse(clase_ternaria %in% PARAM$CN$train$positivos, 1.0000001, 1.0)
    ],
    free_raw_data = FALSE
  )

  dvalid <- lgb.Dataset(
    data = data.matrix(dataset[foto_mes %in% PARAM$CN$train$validation, campos_buenos, with = FALSE]),
    label = dataset[foto_mes %in% PARAM$CN$train$validation, clase01],
    weight = dataset[
      foto_mes %in% PARAM$CN$train$validation,
      ifelse( clase_ternaria %in% PARAM$CN$train$positivos, 1.0000001, 1.0)
    ],
    free_raw_data = FALSE
  )


  param <- list(
    objective = "binary",
    metric = "custom",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    verbosity = -100,
    seed = canaritos_semilla,
    max_depth = -1, # -1 significa no limitar,  por ahora lo dejo fijo
    min_gain_to_split = 0.0, # por ahora, lo dejo fijo
    lambda_l1 = 0.0, # por ahora, lo dejo fijo
    lambda_l2 = 0.0, # por ahora, lo dejo fijo
    max_bin = 31, # por ahora, lo dejo fijo
    num_iterations = 9999, # un numero grande, lo limita early_stopping_rounds
    force_row_wise = TRUE, # para que los alumnos no se atemoricen con  warning
    learning_rate = 0.065,
    feature_fraction = 1.0, # lo seteo en 1
    min_data_in_leaf = 260,
    num_leaves = 60,
    early_stopping_rounds = 200,
    num_threads = 1
  )

  set.seed(canaritos_semilla, kind = "L'Ecuyer-CMRG")
  modelo <- lgb.train(
    data = dtrain,
    valids = list(valid = dvalid),
    eval = fganancia_lgbm_meseta,
    param = param,
    verbose = -100
  )

  tb_importancia <- lgb.importance(model = modelo)
  tb_importancia[, pos := .I]

  fwrite(tb_importancia,
    file = paste0("impo_", GVEZ, ".txt"),
    sep = "\t"
  )

  GVEZ <<- GVEZ + 1

  umbral <- tb_importancia[
    Feature %like% "canarito",
    median(pos) + canaritos_desvios * sd(pos)
  ] # Atencion corto en la mediana mas desvios!!

  col_utiles <- tb_importancia[
    pos < umbral & !(Feature %like% "canarito"),
    Feature
  ]

  col_utiles <- unique(c(
    col_utiles,
    c(campitos, "mes")
  ))

  col_inutiles <- setdiff(colnames(dataset), col_utiles)

  dataset[, (col_inutiles) := NULL]

  cat( "fin CanaritosAsesinos()\n")

  return( tb_importancia )
}


In [46]:
# Estos DOS parametros son los que se deben modificar
PARAM$CN$ratio <- 0.2
PARAM$CN$desvios <- 2


# Parametros quasi fijos
# Parametros de un LightGBM que se genera para estimar la column importance
PARAM$CN$train$clase01_valor1 <- c( "BAJA+2", "BAJA+1")
PARAM$CN$train$positivos <- c( "BAJA+2")
PARAM$CN$train$training <- c( 202101, 202102, 202103)
PARAM$CN$train$validation <- c( 202105 )
PARAM$CN$train$undersampling <- 0.1
PARAM$CN$train$gan1 <- 0.975
PARAM$CN$train$gan0 <- -0.025

In [47]:
if (PARAM$FS$selector == "boruta") {
  # ==== BORUTA: seleccion de variables "a lo Canaritos" ====
  #  se ejecuta solo si  PARAM$FS$selector == "boruta"
  #  adaptado de v950-workflow-01-boruta-014
  
  if( !require("Boruta")) install.packages("Boruta", repos = "http://cran.us.r-project.org")
  require("Boruta")
  
  # ranger es un backend de Random Forest mucho mas rapido; si esta instalado se usa solo
  if( !require("ranger")) install.packages("ranger", repos = "http://cran.us.r-project.org")
  require("ranger")
  
  # Boruta usa la config de entrenamiento de CN; si no existe, la definimos
  if( is.null(PARAM$CN$train) ) {
    PARAM$CN$train <- list(
      clase01_valor1 = c("BAJA+2", "BAJA+1"),
      training       = c(202101, 202102, 202103)
    )
  }
  
  BorutaSelector <- function(boruta_semilla = PARAM$semilla_primigenia) {
  
    cat( "inicio BorutaSelector()\n")
    gc(verbose= FALSE)
  
    # 0) target binaria consistente con CN
    dataset[, clase01 := 0L ]
    dataset[ clase_ternaria %in% PARAM$CN$train$clase01_valor1, clase01 := 1L ]
    dataset[, clase01 := factor(clase01, levels = c(0,1))]
  
    # 1) training + undersampling (igual que Canaritos)
    set.seed(boruta_semilla, kind = "L'Ecuyer-CMRG")
    azar <- runif(nrow(dataset))
    dt_tr_bor <- copy(
      dataset[
        foto_mes %in% PARAM$CN$train$training &
        (clase01 == 1 | azar < PARAM$BR$undersampling)
      ]
    )
  
    # 2) predictores (sin keys ni clases)
    campitos  <- c("numero_de_cliente","foto_mes","clase_ternaria","clase01")
    pred_cols <- setdiff(names(dt_tr_bor), campitos)
  
    # 2.a) quitar columnas constantes o con >95% NA (ahorra RAM/tiempo)
    const <- pred_cols[vapply(pred_cols,
      function(v) data.table::uniqueN(dt_tr_bor[[v]], na.rm=TRUE) <= 1L, logical(1))]
    hi_na <- pred_cols[vapply(pred_cols,
      function(v) mean(is.na(dt_tr_bor[[v]])) > 0.95, logical(1))]
    pred_cols <- setdiff(pred_cols, union(const, hi_na))
  
    # 2.b) tipos: char -> factor
    for (v in pred_cols) if (is.character(dt_tr_bor[[v]])) dt_tr_bor[, (v) := factor(get(v))]
  
    # 2.c) imputacion simple SOLO para Boruta (respetando tipo original)
    for (v in pred_cols) {
      col <- dt_tr_bor[[v]]
  
      if (is.integer(col)) {
        m <- suppressWarnings(median(as.numeric(col), na.rm = TRUE))
        if (!is.finite(m)) m <- 0
        set(dt_tr_bor, which(is.na(col)), v, as.integer(round(m)))
  
      } else if (is.numeric(col)) {
        m <- suppressWarnings(median(col, na.rm = TRUE))
        if (!is.finite(m)) m <- 0
        set(dt_tr_bor, which(is.na(col)), v, m)
  
      } else if (is.factor(col)) {
        if (anyNA(col)) {
          dt_tr_bor[, (v) := fcase(is.na(get(v)), "Missing", default = as.character(get(v)))]
          dt_tr_bor[, (v) := factor(get(v))]
        }
      }
    }
  
    # 3) pesos de clase (OPCIONAL, default vanilla = sin pesos,
    #    igual que Canaritos que solo hace undersampling)
    if( PARAM$BR$classweights ) {
      n0 <- sum(dt_tr_bor$clase01 == 0)
      n1 <- sum(dt_tr_bor$clase01 == 1)
      class_w <- c("0" = 1, "1" = ifelse(n1 > 0, n0/n1, 1))
    } else {
      class_w <- NULL
    }
  
    # 4) BORUTA (detecta backend y pasa class weights solo si corresponde)
    use_ranger <- requireNamespace("ranger", quietly = TRUE)
  
    bor_args <- list(
      x           = as.data.frame(dt_tr_bor[, ..pred_cols]),
      y           = dt_tr_bor$clase01,
      doTrace     = 2,
      ntree       = PARAM$BR$ntree,
      maxRuns     = PARAM$BR$maxRuns,
      pValue      = PARAM$BR$pValue,
      mcAdj       = PARAM$BR$mcAdj,
      getImp      = getImpRfZ,
      holdHistory = PARAM$BR$holdHistory
      # max.depth   = 16
    )
    if( PARAM$BR$threads > 0L ) bor_args$num.threads <- PARAM$BR$threads
    if( !is.null(class_w) ) {
      if (use_ranger) bor_args$class.weights <- class_w else bor_args$classwt <- class_w
    }
  
    set.seed(boruta_semilla, kind = "L'Ecuyer-CMRG")
    bor_fit  <- do.call(Boruta, bor_args)
    # TentativeRoughFix exige holdHistory=TRUE;
    #  si se corrio sin historia, los tentative simplemente quedan afuera
    if( PARAM$BR$holdHistory ) {
      bor_fix <- TentativeRoughFix(bor_fit)
    } else {
      bor_fix <- bor_fit
    }
  
    # 5) resultados y persistencias
    bor_stats <- as.data.table(attStats(bor_fix), keep.rownames = "variable")
    sel_vars  <- getSelectedAttributes(bor_fix, withTentative = FALSE)
  
    fwrite(bor_stats,                    "boruta_stats.txt", sep = "\t")
    fwrite(data.table(variable=sel_vars),"boruta.txt",      sep = "\t")
  
    cat(sprintf("BORUTA: candidatos=%d  seleccionados=%d\n",
      length(pred_cols), length(sel_vars)))
  
    # 6) recorte del dataset GLOBAL, igual que hace CanaritosAsesinos
    keep_cols <- unique(c(campitos, sel_vars))
    keep_cols <- intersect(keep_cols, names(dataset))
    dataset[, (setdiff(names(dataset), keep_cols)) := NULL]
  
    cat( "fin BorutaSelector()\n")
  
    invisible( bor_stats )
  }
} else {
  cat("skip Boruta install/define - canarito only\n")
}


skip Boruta install/define - canarito only


In [48]:
# seleccion de variables, segun  PARAM$FS$selector  (canarito | boruta)
if( PARAM$FS$selector == "boruta" ) {
  cat("Seleccion de variables: BORUTA\n")
  tb_boruta <- BorutaSelector(
    boruta_semilla = PARAM$semilla_primigenia
  )
} else {
  cat("Seleccion de variables: CANARITOS ASESINOS\n")
  tb_importancia <- CanaritosAsesinos(
    canaritos_ratio = PARAM$CN$ratio,
    canaritos_desvios = PARAM$CN$desvios,
    canaritos_semilla = PARAM$semilla_primigenia
  )
}


Seleccion de variables: CANARITOS ASESINOS
inicio CanaritosAsesinos()
fin CanaritosAsesinos()


In [49]:
# grabo la importancia, ver el archivo directamente en la carpeta
if( PARAM$FS$selector == "canarito" ) {
  fwrite( tb_importancia,
    file="canaritos.txt",
    sep="\t"
  )
}


In [50]:
# verifico
ncol(dataset)
colnames(dataset)

[1] 444

[1] "numero_de_cliente"                               
  [2] "foto_mes"                                        
  [3] "internet"                                        
  [4] "cliente_edad"                                    
  [5] "cliente_antiguedad"                              
  [6] "cproductos"                                      
  [7] "ccaja_ahorro"                                    
  [8] "cdescubierto_preacordado"                        
  [9] "ctarjeta_debito_transacciones"                   
 [10] "ctarjeta_visa"                                   
 [11] "ctarjeta_visa_transacciones"                     
 [12] "ctarjeta_master_transacciones"                   
 [13] "cprestamos_personales"                           
 [14] "ccaja_seguridad"                                 
 [15] "cpayroll_trx"                                    
 [16] "ccuenta_debitos_automaticos"                     
 [17] "ctarjeta_visa_debitos_automaticos"               
 [18] "ccomisiones_otras"                               
 [19] "ctransferencias_recibidas"                       
 [20] "ctransferencias_emitidas"                        
 [21] "cextraccion_autoservicio"                        
 [22] "tcallcenter"                                     
 [23] "thomebanking"                                    
 [24] "chomebanking_transacciones"                      
 [25] "ccajas_consultas"                                
 [26] "ctrx_quarter"                                    
 [27] "Master_status"                                   
 [28] "Master_Fvencimiento"                             
 [29] "Master_fultimo_cierre"                           
 [30] "Master_fechaalta"                                
 [31] "Master_cconsumos"                                
 [32] "Visa_status"                                     
 [33] "Visa_Fvencimiento"                               
 [34] "Visa_fechaalta"                                  
 [35] "Visa_cconsumos"                                  
 [36] "clase_ternaria"                                  
 [37] "mrentabilidad_rank"                              
 [38] "mrentabilidad_annual_rank"                       
 [39] "mcomisiones_rank"                                
 [40] "mactivos_margen_rank"                            
 [41] "mpasivos_margen_rank"                            
 [42] "mcuenta_corriente_rank"                          
 [43] "mcaja_ahorro_rank"                               
 [44] "mcaja_ahorro_dolares_rank"                       
 [45] "mcuentas_saldo_rank"                             
 [46] "mautoservicio_rank"                              
 [47] "mtarjeta_visa_consumo_rank"                      
 [48] "mtarjeta_master_consumo_rank"                    
 [49] "mprestamos_personales_rank"                      
 [50] "mplazo_fijo_dolares_rank"                        
 [51] "minversion2_rank"                                
 [52] "mpayroll_rank"                                   
 [53] "mcuenta_debitos_automaticos_rank"                
 [54] "mttarjeta_visa_debitos_automaticos_rank"         
 [55] "mpagomiscuentas_rank"                            
 [56] "mcomisiones_mantenimiento_rank"                  
 [57] "mcomisiones_otras_rank"                          
 [58] "mtransferencias_recibidas_rank"                  
 [59] "Master_mfinanciacion_limite_rank"                
 [60] "Master_msaldototal_rank"                         
 [61] "Master_msaldopesos_rank"                         
 [62] "Master_msaldodolares_rank"                       
 [63] "Master_mconsumosdolares_rank"                    
 [64] "Master_mlimitecompra_rank"                       
 [65] "Master_mpagosdolares_rank"                       
 [66] "Master_mpagominimo_rank"                         
 [67] "Visa_mfinanciacion_limite_rank"                  
 [68] "Visa_msaldototal_rank"                           
 [69] "Visa_msaldopesos_rank"                           
 [70] "Visa_msaldodolares_rank"                         
 [71] "Visa_

### 9.7.2 Modelado

#### 9.7.2.1 Training Strategy


Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ] sin undersampling de los CONTINUA

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 20% de los CONTINUA

In [51]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)

PARAM$trainingstrategy$training_pct <- 0.2


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [52]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [53]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [54]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [55]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 164479

####  9.7.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Cantidad de iteraciones inteligentes de la Optimizacion Bayesiana = **40**

In [56]:
# paquetes necesarios para la Bayesian Optimization
if(!require("DiceKriging")) install.packages("DiceKriging")
require("DiceKriging")

if(!require("mlrMBO")) install.packages("mlrMBO")
require("mlrMBO")

Loading required package: DiceKriging



Loading required package: mlrMBO



Loading required package: mlr



Loading required package: ParamHelpers




Attaching package: ‘ParamHelpers’




...


Attaching package: ‘checkmate’




The following object is masked from ‘package:DiceKriging’:

    checkNames




The following object is masked from ‘package:R.utils’:

    asInt





Attaching package: ‘smoof’




The following objects are masked from ‘package:R.oo’:

    getDescription, getName




Definición de la Bayesian Optimization
<br> Si se desea optimizar un hiperparámetro que esta como fijo, debe QUITARSE de param_fijos y agregarse a PARAM$hipeparametertuning$hs

In [57]:
# 40 es el minimo razonable,  un buen analista Sr aumentara este valor
PARAM$hipeparametertuning$num_interations <- 30

# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  extra_trees= FALSE,

  num_leaves= 128,
  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0.0, # min_gain_to_split >= 0.0
  min_data_in_leaf= 0,
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  max_bin= 31,
  early_stopping_rounds= 0
)



In [58]:
PARAM$hipeparametertuning$hs <- makeParamSet(
  makeNumericParam("num_iterations", lower= 8, upper= 12, trafo= function(x) round(2^x) ), # 256..4096 was 2..10 (4..1024) - covers HT 1018..3000
  makeNumericParam("num_leaves", lower= 5, upper= 8, trafo= function(x) round(2^x) ), # 32..256 FIX was 4..2048 (2040 unrealistic)
  makeNumericParam("learning_rate", lower= -6, upper= -3.2, trafo= function(x) 2^x ), # 0.015..0.108 was -7..-0.1 (0.007..0.93) - narrow around HT 0.02-0.05
  makeNumericParam("feature_fraction", lower= 0.4, upper= 0.8), # was 0.05..0.90 - HT optimum 0.6
  makeNumericParam("min_data_in_leaf", lower= 4, upper= 10, trafo= function(x) round(2^x) ), # 16..1024 NEW from HT 30..400
  makeNumericParam("lambda_l2", lower= 4, upper= 9, trafo= function(x) 2^x), # 16..512 NEW from HT 100..300
  makeNumericParam("bagging_fraction", lower= 0.7, upper= 1.0) # NEW from HT 0.7..1
)


Función "señora caja negra"  que es llamada para verificar la realidad por la Bayesian Optimization

In [59]:
# En  x llegan los parmaetros de la bayesiana
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

EstimarGanancia_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[x$num_iterations]]


  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " AUC ", AUC
  )

  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return(AUC)
}

seteo de la Bayesian Optimization (complejo)
<br> copiado y pegado de la documentación de la librería

In [60]:
configureMlr(show.learner.output = FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo
obj.fun <- makeSingleObjectiveFunction(
    fn= EstimarGanancia_AUC_lightgbm, # la funcion que voy a maximizar
    minimize= FALSE, # estoy Maximizando AUC
    noisy= FALSE,
    par.set= PARAM$hipeparametertuning$hs,
    has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
    save.on.disk.at.time= 600,
    save.file.path= "HT.RDATA"
)

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
    ctrl,
    iters= PARAM$hipeparametertuning$num_interations  # cantidad de iteraciones inteligentes
)

# defino el método estandar para la creacion de los puntos iniciales
#   los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit = makeMBOInfillCritEI())

# mas configuraciones
surr.km <- makeLearner(
    "regr.km",
    predict.type= "se",
    covtype= "matern3_2",
    control= list(trace = TRUE)
)

Corrida de la Bayesian Optimization,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer

Si corrío a medias y llegó a las iteraciones inteligentes, en el archivo binario HT.RDATA quedó lo ya procesado y es utilizado para retomar la corrida desde lo último que llegó a grabar.

In [61]:
# inicio la optimizacion bayesiana - warm-start DESACTIVADO temporalmente, solo hs fixeada 32..256
if (!file.exists("HT.RDATA")) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue("HT.RDATA")
}


Computing y column(s) for design. Not provided.



Thu Sep 03 21:23:12 2026  2685, 106, 0.0363221281333659, 0.670158835119676, 186, 226.107785287202, 0.886671091462922 AUC 0.936619603154568



Thu Sep 03 21:27:18 2026  1300, 81, 0.0270718314596404, 0.600542681105107, 105, 33.000255686248, 0.82027048456329 AUC 0.938294366843666



Thu Sep 03 21:28:46 2026  547, 87, 0.0332603528740547, 0.722111742976611, 38, 65.5887484409189, 0.803044241057523 AUC 0.938055719622621



Thu Sep 03 21:32:58 2026  1641, 96, 0.0484764132862978, 0.63446499378956, 163, 115.195357474173, 0.86591978895941 AUC 0.935928038017846



...

[mbo] 0: num_iterations=2.91e+03; num_leaves=35; learning_rate=0.0698; feature_fraction=0.513; min_data_in_leaf=607; lambda_l2=340; bagging_fraction=0.756 : y = 0.936 : 652.2 secs : initdesign



[mbo] 0: num_iterations=332; num_leaves=128; learning_rate=0.0338; feature_fraction=0.423; min_data_in_leaf=46; lambda_l2=493; bagging_fraction=0.925 : y = 0.935 : 83.5 secs : initdesign



[mbo] 0: num_iterations=3.19e+03; num_leaves=103; learning_rate=0.0401; feature_fraction=0.751; min_data_in_leaf=56; lambda_l2=35.7; bagging_fraction=0.86 : y = 0.936 : 400.0 secs : initdesign



Saved the current state after iteration 1 in the file HT.RDATA.



Thu Sep 03 23:38:02 2026  1646, 224, 0.0203098123262139, 0.428155423909685, 685, 114.843213018843, 0.808230090613805 AUC 0.938871837507236



la bayesian optimization ha corrido, extraigo los mejores hiperparametros

In [ ]:
# almaceno los resultados de la Bayesian Optimization
# y capturo los mejores hiperparametros encontrados

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)

# ordeno en forma descendente por AUC = y
setorder(tb_bayesiana, -y, -num_iterations)

# grabo para eventualmente poder utilizarlos en OTRA corrida
fwrite( tb_bayesiana,
  file="BO_log.txt",
  sep="\t"
)

# los mejores hiperparámetros son los que quedaron en el registro 1 de la tabla
PARAM$out$lgbm$mejores_hiperparametros <- tb_bayesiana[
  1, # el primero es el de mejor AUC
  setdiff(colnames(tb_bayesiana),
    c("y","dob","eol","error.message","exec.time","ei","error.model",
      "train.time","prop.type","propose.time","se","mean","iter")),
  with= FALSE
]

print(PARAM$out$lgbm$mejores_hiperparametros)

### 9.7.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización bayesiana

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL
fijos$num_leaves <- NULL
fijos$min_data_in_leaf <- NULL
fijos$lambda_l2 <- NULL
fijos$bagging_fraction <- NULL
fijos$min_sum_hessian_in_leaf <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- copy(fijos)

x <- PARAM$out$lgbm$mejores_hiperparametros
param_final$num_iterations <- round( 2^x$num_iterations )
param_final$num_leaves <- round( 2^x$num_leaves )
param_final$learning_rate <- 2^x$learning_rate
param_final$feature_fraction <- x$feature_fraction
param_final$min_data_in_leaf <- round( 2^x$min_data_in_leaf )
param_final$lambda_l2 <- 2^x$lambda_l2
param_final$bagging_fraction <- x$bagging_fraction



In [ ]:
param_final

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
# poner este valor en 5 o en 1  si se está muy ajustado de tiempo

PARAM$FT$semillerio <- 10  # cantidad de semillas

In [ ]:
if(!require("primes")) install.packages("primes")
require("primes")

In [ ]:
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
# me quedo con PARAM$semillerio  primos al azar
PARAM$FT$semillas <- sample(primos)[seq(PARAM$FT$semillerio)]

cat( PARAM$FT$semillas)

In [ ]:
primero <- TRUE

crear_modelo_final <- function( sem ) {

  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  if( !file.exists(nombre_arch) )
  {
    param_final$seed <- sem

    set.seed(sem, kind = "L'Ecuyer-CMRG")
    final_model <- lgb.train(
      data= dfinal_train,
      param= param_final,
      verbose= -100
    )

    lgb.save(final_model, nombre_arch) # grabo el modelo"

    # grabo la primer importancia de variables
    #  Natalia : da lo mismo cual se guarda
    if( primero)
    {
      primero <<- FALSE
      tb_importancia <- as.data.table(lgb.importance(final_model))
      archivo_importancia <- "impo.txt"

      fwrite( tb_importancia,
        file= archivo_importancia,
        sep= "\t"
      )
    }

    rm(final_model)
    gc(full = TRUE, verbose=FALSE)
  }
}

In [ ]:
gc(full = TRUE, verbose=FALSE)
dir.create("modelos", showWarnings =FALSE)

primero <- TRUE
for( sem in PARAM$FT$semillas)  crear_modelo_final(sem)

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [ ]:
# aplico final_model   a dfuture

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := 0]

datos_matrix <- data.matrix(dfuture[, campos_buenos, with= FALSE])

In [ ]:
agregar_a_ensemble <- function( sem ) {

  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  final_model <- lgb.load(nombre_arch)

  prediccion <- predict(
    final_model,
    datos_matrix
  )

  tb_prediccion[, paste0("prob_", isem) := prediccion]
  tb_prediccion[, prob := prob + prediccion]

  rm(final_model)
  rm(prediccion)
  gc(full = TRUE, verbose=FALSE)
}

In [ ]:
for( isem in seq(length(PARAM$FT$semillas)) )
{
  sem <- PARAM$FT$semillas[ isem ]
  agregar_a_ensemble( sem )
  gc(full = TRUE, verbose=FALSE)
}

rm( datos_matrix)
gc(full = TRUE, verbose=FALSE)

tb_prediccion[, prob := prob /length(PARAM$FT$semillas) ]

In [ ]:
# veo que hay en tb_prediccion
tb_prediccion

In [ ]:
# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggles

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "data-mining-senior-2026-b"
PARAM$kaggle$cortes <- seq(8000, 12000, by = 500)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0(
    "./kaggle/KA",
    PARAM$experimento, "_",
    "s", length(PARAM$FT$semillas), "_",
    envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")